# *Staphylococcus aureus* USA300 WGS Variant Calling Pipeline

End-to-end whole-genome sequencing variant-calling workflow implemented in Google Colab using verified USA300_FPR3757 reference and ERR17521341 paired-end Illumina WGS data.

**Workflow:** FastQC → Trimmomatic → BWA-MEM → SAMtools → FreeBayes → variant filtering

**Reference:** USA300_FPR3757, GCF_000013465.1

**Dataset:** ERR17521341


In [32]:
%%bash
mkdir -p /content/reads
cd /content/reads
prefetch ERR17521341
fastq-dump --split-files --gzip ERR17521341
ls -lh

2026-08-16T08:10:48 prefetch.3.4.1: 1) Resolving 'ERR17521341'...
2026-08-16T08:10:48 prefetch.3.4.1: Current preference is set to retrieve SRA Normalized Format files with full base quality scores
2026-08-16T08:10:48 prefetch.3.4.1: 1) Downloading 'ERR17521341'...
2026-08-16T08:10:48 prefetch.3.4.1:  SRA Normalized Format file is being retrieved
2026-08-16T08:10:48 prefetch.3.4.1:  Downloading via HTTPS...
2026-08-16T08:10:56 prefetch.3.4.1:  HTTPS download succeed
2026-08-16T08:10:57 prefetch.3.4.1:  'ERR17521341' is valid: 499663732 bytes were streamed from 499661445
2026-08-16T08:10:57 prefetch.3.4.1: 1) 'ERR17521341' was downloaded successfully
2026-08-16T08:10:57 prefetch.3.4.1: 1) Resolving 'ERR17521341's dependencies...
2026-08-16T08:10:57 prefetch.3.4.1: 'ERR17521341' has 0 unresolved dependencies
Read 6167891 spots for ERR17521341
Written 6167891 spots for ERR17521341
total 783M
drwxr-xr-x 2 root root 4.0K Aug 16 08:10 ERR17521341
-rw-r--r-- 1 root root 392M Aug 16 08:21 ERR1

In [34]:
%%bash
mkdir -p /content/reference
cd /content/reference

# Download USA300_FPR3757 reference genome from NCBI
wget -q "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/013/465/GCF_000013465.1_MSSA476/GCF_000013465.1_MSSA476_genomic.fna.gz" -O USA300_FPR3757.fna.gz

# Decompress it
gunzip USA300_FPR3757.fna.gz

# Check it downloaded correctly
ls -lh
head -5 USA300_FPR3757.fna

total 0
-rw-r--r-- 1 root root 0 Aug 16 08:22 USA300_FPR3757.fna.gz



gzip: USA300_FPR3757.fna.gz: unexpected end of file
head: cannot open 'USA300_FPR3757.fna' for reading: No such file or directory


CalledProcessError: Command 'b'mkdir -p /content/reference\ncd /content/reference\n\n# Download USA300_FPR3757 reference genome from NCBI\nwget -q "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/013/465/GCF_000013465.1_MSSA476/GCF_000013465.1_MSSA476_genomic.fna.gz" -O USA300_FPR3757.fna.gz\n\n# Decompress it\ngunzip USA300_FPR3757.fna.gz\n\n# Check it downloaded correctly\nls -lh\nhead -5 USA300_FPR3757.fna\n'' returned non-zero exit status 1.

In [35]:
%%bash
cd /content/reference

# Correct accession for USA300_FPR3757
wget -q "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/013/465/GCF_000013465.1_USA300_FPR3757/GCF_000013465.1_USA300_FPR3757_genomic.fna.gz" -O USA300_FPR3757.fna.gz

ls -lh

total 0
-rw-r--r-- 1 root root 0 Aug 16 08:22 USA300_FPR3757.fna.gz


In [36]:
%%bash
mkdir -p /content/reference
cd /content/reference

wget "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/013/465/GCF_000013465.1_ASM1346v1/GCF_000013465.1_ASM1346v1_genomic.fna.gz" -O USA300_FPR3757.fna.gz

ls -lh

total 828K
-rw-r--r-- 1 root root 828K Apr 12  2020 USA300_FPR3757.fna.gz


--2026-08-16 08:22:39--  https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/013/465/GCF_000013465.1_ASM1346v1/GCF_000013465.1_ASM1346v1_genomic.fna.gz
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.31, 130.14.250.7, 2607:f220:41e:250::31, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.31|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 847833 (828K) [application/x-gzip]
Saving to: ‘USA300_FPR3757.fna.gz’

     0K .......... .......... .......... .......... ..........  6% 9.16M 0s
    50K .......... .......... .......... .......... .......... 12% 19.7M 0s
   100K .......... .......... .......... .......... .......... 18%  163M 0s
   150K .......... .......... .......... .......... .......... 24% 19.0M 0s
   200K .......... .......... .......... .......... .......... 30%  136M 0s
   250K .......... .......... .......... .......... .......... 36%  186M 0s
   300K .......... .......... .......... .......... .......... 42

In [37]:
%%bash
cd /content/reference
gunzip USA300_FPR3757.fna.gz
ls -lh
head -5 USA300_FPR3757.fna

total 2.9M
-rw-r--r-- 1 root root 2.9M Apr 12  2020 USA300_FPR3757.fna
>NC_007793.1 Staphylococcus aureus subsp. aureus USA300_FPR3757, complete sequence
ACTACTGCTCAATTTTTTTACTTTTATCGATTAAAGATAGAAATACACGATGCGAGCAATCAAATTTCATAACATCACCA
TGAGTTTGGTCCGAAGCATGAGTGTTTACAATGTTCGAACACCTTATACAGTTCTTATACATACTTTATAAATTATTTCC
CAAACTGTTTTGATACACTCACTAACAGATACTCTATAGAAGGAAAAGTTATCCACTTATGCACATTTATAGTTTTCAGA
ATTGTGGATAATTAGAAATTACACACAAAGTTATACTATTTTTAGCAACATATTCACAGGTATTTGACATATAGAGAACT


In [38]:
%%bash
cd /content/reference
bwa index USA300_FPR3757.fna
ls -lh

total 7.8M
-rw-r--r-- 1 root root 2.9M Apr 12  2020 USA300_FPR3757.fna
-rw-r--r-- 1 root root   12 Aug 16 08:22 USA300_FPR3757.fna.amb
-rw-r--r-- 1 root root  452 Aug 16 08:22 USA300_FPR3757.fna.ann
-rw-r--r-- 1 root root 2.8M Aug 16 08:22 USA300_FPR3757.fna.bwt
-rw-r--r-- 1 root root 713K Aug 16 08:22 USA300_FPR3757.fna.pac
-rw-r--r-- 1 root root 1.4M Aug 16 08:22 USA300_FPR3757.fna.sa


[bwa_index] Pack FASTA... 0.02 sec
[bwa_index] Construct BWT for the packed sequence...
[bwa_index] 0.83 seconds elapse.
[bwa_index] Update BWT... 0.02 sec
[bwa_index] Pack forward-only FASTA... 0.01 sec
[bwa_index] Construct SA from BWT and Occ... 0.44 sec
[main] Version: 0.7.17-r1188
[main] CMD: bwa index USA300_FPR3757.fna
[main] Real time: 1.387 sec; CPU: 1.322 sec


In [39]:
%%bash
mkdir -p /content/fastqc_raw
fastqc /content/reads/ERR17521341_1.fastq.gz \
       /content/reads/ERR17521341_2.fastq.gz \
       -o /content/fastqc_raw/ \
       --threads 2
ls -lh /content/fastqc_raw/



[0.027s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.027s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Analysis complete for ERR17521341_2.fastq.gz
Analysis complete for ERR17521341_1.fastq.gz
total 2.5M
-rw-r--r-- 1 root root 575K Aug 16 08:24 ERR17521341_1_fastqc.html
-rw-r--r-- 1 root root 658K Aug 16 08:24 ERR17521341_1_fastqc.zip
-rw-r--r-- 1 root root 581K Aug 16 08:24 ERR17521341_2_fastqc.html
-rw-r--r-- 1 root root 662K Aug 16 08:24 ERR17521341_2_fastqc.zip


Started analysis of ERR17521341_1.fastq.gz
Started analysis of ERR17521341_2.fastq.gz
Approx 5% complete for ERR17521341_1.fastq.gz
Approx 5% complete for ERR17521341_2.fastq.gz
Approx 10% complete for ERR17521341_1.fastq.gz
Approx 10% complete for ERR17521341_2.fastq.gz
Approx 15% complete for ERR17521341_2.fastq.gz
Approx 15% complete for ERR17521341_1.fastq.gz
Approx 20% complete for ERR17521341_2.fastq.gz
Approx 20% complete for ERR17521341_1.fastq.gz
Approx 25% complete for ERR17521341_2.fastq.gz
Approx 25% complete for ERR17521341_1.fastq.gz
Approx 30% complete for ERR17521341_2.fastq.gz
Approx 30% complete for ERR17521341_1.fastq.gz
Approx 35% complete for ERR17521341_2.fastq.gz
Approx 35% complete for ERR17521341_1.fastq.gz
Approx 40% complete for ERR17521341_2.fastq.gz
Approx 40% complete for ERR17521341_1.fastq.gz
Approx 45% complete for ERR17521341_2.fastq.gz
Approx 45% complete for ERR17521341_1.fastq.gz
Approx 50% complete for ERR17521341_2.fastq.gz
Approx 50% complete for

In [40]:
%%bash
unzip -p /content/fastqc_raw/ERR17521341_1_fastqc.zip ERR17521341_1_fastqc/summary.txt
echo "---"
unzip -p /content/fastqc_raw/ERR17521341_2_fastqc.zip ERR17521341_2_fastqc/summary.txt

PASS	Basic Statistics	ERR17521341_1.fastq.gz
PASS	Per base sequence quality	ERR17521341_1.fastq.gz
FAIL	Per tile sequence quality	ERR17521341_1.fastq.gz
PASS	Per sequence quality scores	ERR17521341_1.fastq.gz
PASS	Per base sequence content	ERR17521341_1.fastq.gz
WARN	Per sequence GC content	ERR17521341_1.fastq.gz
PASS	Per base N content	ERR17521341_1.fastq.gz
PASS	Sequence Length Distribution	ERR17521341_1.fastq.gz
WARN	Sequence Duplication Levels	ERR17521341_1.fastq.gz
PASS	Overrepresented sequences	ERR17521341_1.fastq.gz
PASS	Adapter Content	ERR17521341_1.fastq.gz
---
PASS	Basic Statistics	ERR17521341_2.fastq.gz
PASS	Per base sequence quality	ERR17521341_2.fastq.gz
WARN	Per tile sequence quality	ERR17521341_2.fastq.gz
PASS	Per sequence quality scores	ERR17521341_2.fastq.gz
PASS	Per base sequence content	ERR17521341_2.fastq.gz
WARN	Per sequence GC content	ERR17521341_2.fastq.gz
PASS	Per base N content	ERR17521341_2.fastq.gz
PASS	Sequence Length Distribution	ERR17521341_2.fastq.gz
WARN

In [41]:
%%bash
mkdir -p /content/trimmed

trimmomatic PE \
  /content/reads/ERR17521341_1.fastq.gz \
  /content/reads/ERR17521341_2.fastq.gz \
  /content/trimmed/R1_paired.fastq.gz \
  /content/trimmed/R1_unpaired.fastq.gz \
  /content/trimmed/R2_paired.fastq.gz \
  /content/trimmed/R2_unpaired.fastq.gz \
  LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36

ls -lh /content/trimmed/

total 0


bash: line 3: trimmomatic: command not found


In [42]:
%%bash
ls /usr/share/trimmomatic/

NexteraPE-PE.fa
TruSeq2-PE.fa
TruSeq2-SE.fa
TruSeq3-PE-2.fa
TruSeq3-PE.fa
TruSeq3-SE.fa


In [43]:
%%bash
find / -name "trimmomatic*.jar" 2>/dev/null

/usr/share/java/trimmomatic.jar
/usr/share/java/trimmomatic-0.39.jar


CalledProcessError: Command 'b'find / -name "trimmomatic*.jar" 2>/dev/null\n'' returned non-zero exit status 1.

In [44]:
%%bash
mkdir -p /content/trimmed

java -jar /usr/share/java/trimmomatic.jar PE \
  /content/reads/ERR17521341_1.fastq.gz \
  /content/reads/ERR17521341_2.fastq.gz \
  /content/trimmed/R1_paired.fastq.gz \
  /content/trimmed/R1_unpaired.fastq.gz \
  /content/trimmed/R2_paired.fastq.gz \
  /content/trimmed/R2_unpaired.fastq.gz \
  LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36

ls -lh /content/trimmed/

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
total 774M
-rw-r--r-- 1 root root 386M Aug 16 08:32 R1_paired.fastq.gz
-rw-r--r-- 1 root root 2.9M Aug 16 08:32 R1_unpaired.fastq.gz
-rw-r--r-- 1 root root 385M Aug 16 08:32 R2_paired.fastq.gz
-rw-r--r-- 1 root root 1.1M Aug 16 08:32 R2_unpaired.fastq.gz


TrimmomaticPE: Started with arguments:
 /content/reads/ERR17521341_1.fastq.gz /content/reads/ERR17521341_2.fastq.gz /content/trimmed/R1_paired.fastq.gz /content/trimmed/R1_unpaired.fastq.gz /content/trimmed/R2_paired.fastq.gz /content/trimmed/R2_unpaired.fastq.gz LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36
Multiple cores found: Using 2 threads
Quality encoding detected as phred33
Input Read Pairs: 6167891 Both Surviving: 6108306 (99.03%) Forward Only Surviving: 39210 (0.64%) Reverse Only Surviving: 13510 (0.22%) Dropped: 6865 (0.11%)
TrimmomaticPE: Completed successfully


In [45]:
%%bash
mkdir -p /content/aligned

bwa mem \
  -t 2 \
  -R "@RG\tID:ERR17521341\tSM:USA300\tPL:ILLUMINA\tLB:lib1" \
  /content/reference/USA300_FPR3757.fna \
  /content/trimmed/R1_paired.fastq.gz \
  /content/trimmed/R2_paired.fastq.gz \
  > /content/aligned/aligned.sam

ls -lh /content/aligned/

total 4.9G
-rw-r--r-- 1 root root 4.9G Aug 16 08:44 aligned.sam


[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::process] read 133634 sequences (20000032 bp)...
[M::process] read 133572 sequences (20000238 bp)...
[M::mem_pestat] # candidate unique pairs for (FF, FR, RF, RR): (24, 63567, 17, 31)
[M::mem_pestat] analyzing insert size distribution for orientation FF...
[M::mem_pestat] (25, 50, 75) percentile: (132, 159, 241)
[M::mem_pestat] low and high boundaries for computing mean and std.dev: (1, 459)
[M::mem_pestat] mean and std.dev: (171.09, 70.88)
[M::mem_pestat] low and high boundaries for proper pairs: (1, 568)
[M::mem_pestat] analyzing insert size distribution for orientation FR...
[M::mem_pestat] (25, 50, 75) percentile: (265, 308, 351)
[M::mem_pestat] low and high boundaries for computing mean and std.dev: (93, 523)
[M::mem_pestat] mean and std.dev: (308.47, 64.54)
[M::mem_pestat] low and high boundaries for proper pairs: (7, 609)
[M::mem_pestat] analyzing insert size distribution for orientation RF...
[M::mem_pestat] (25, 50, 75) percent

In [46]:
%%bash
cd /content/aligned

samtools view -bS aligned.sam -o aligned.bam
samtools sort aligned.bam -o aligned_sorted.bam
samtools index aligned_sorted.bam
samtools flagstat aligned_sorted.bam

12249987 + 0 in total (QC-passed reads + QC-failed reads)
12216612 + 0 primary
0 + 0 secondary
33375 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
12235896 + 0 mapped (99.88% : N/A)
12202521 + 0 primary mapped (99.88% : N/A)
12216612 + 0 paired in sequencing
6108306 + 0 read1
6108306 + 0 read2
12117530 + 0 properly paired (99.19% : N/A)
12193710 + 0 with itself and mate mapped
8811 + 0 singletons (0.07% : N/A)
4 + 0 with mate mapped to a different chr
2 + 0 with mate mapped to a different chr (mapQ>=5)


[bam_sort_core] merging from 5 files and 1 in-memory blocks...


In [47]:
%%bash
mkdir -p /content/variants

freebayes \
  -f /content/reference/USA300_FPR3757.fna \
  --ploidy 1 \
  /content/aligned/aligned_sorted.bam \
  > /content/variants/variants.vcf

echo "Done. Line count:"
wc -l /content/variants/variants.vcf

Done. Line count:
530 /content/variants/variants.vcf


index file /content/reference/USA300_FPR3757.fna.fai not found, generating...


In [48]:
%%bash
# Show header lines (metadata)
grep "^#" /content/variants/variants.vcf | tail -5

echo "---"

# Show first 5 actual variant records
grep -v "^#" /content/variants/variants.vcf | head -5

##FORMAT=<ID=QR,Number=1,Type=Integer,Description="Sum of quality of the reference observations">
##FORMAT=<ID=AO,Number=A,Type=Integer,Description="Alternate allele observation count">
##FORMAT=<ID=QA,Number=A,Type=Integer,Description="Sum of quality of the alternate observations">
##FORMAT=<ID=MIN_DP,Number=1,Type=Integer,Description="Minimum depth in gVCF output block.">
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	USA300
---
NC_007793.1	317	.	A	G	24626.5	.	AB=0;ABP=0;AC=1;AF=1;AN=1;AO=689;CIGAR=1X;DP=689;DPB=689;DPRA=0;EPP=4.67751;EPPR=0;GTI=0;LEN=1;MEANALT=1;MQM=60;MQMR=0;NS=1;NUMALT=1;ODDS=5670.46;PAIRED=0.986938;PAIREDR=0;PAO=0;PQA=0;PQR=0;PRO=0;QA=27454;QR=0;RO=0;RPL=325;RPP=7.80393;RPPR=0;RPR=364;RUN=1;SAF=369;SAP=10.5774;SAR=320;SRF=0;SRP=0;SRR=0;TYPE=snp;technology.ILLUMINA=1	GT:DP:AD:RO:QR:AO:QA:GL	1:689:0,689:0:0:689:27454:-2468.6,0
NC_007793.1	2309	.	G	C	0	.	AB=0;ABP=0;AC=0;AF=0;AN=1;AO=131;CIGAR=1X;DP=843;DPB=843;DPRA=0;EPP=10.3204;EPPR=6.82523;GTI=0;LEN=1;MEANALT=3;MQM

In [49]:
%%bash
# Count total records
echo "Total VCF records:"
grep -v "^#" /content/variants/variants.vcf | wc -l

# Filter: QUAL>20, AF=1 (fixed variants)
echo "High confidence fixed variants (QUAL>20, AF=1):"
grep -v "^#" /content/variants/variants.vcf | \
  awk -F'\t' '$6 > 20 && $8 ~ /AF=1;/' | wc -l

# Show those filtered variants
echo "---"
grep -v "^#" /content/variants/variants.vcf | \
  awk -F'\t' '$6 > 20 && $8 ~ /AF=1;/' | \
  cut -f1,2,4,5,6,8 | \
  grep -oP 'NC_007793.1\t\d+\t\w+\t\w+\t[\d.]+'

Total VCF records:
465
High confidence fixed variants (QUAL>20, AF=1):
65
---
NC_007793.1	317	A	G	24626.5
NC_007793.1	5010	C	G	29210.6
NC_007793.1	34179	T	A	30645
NC_007793.1	41290	ATTTTTTTAGT	ATTTTTTTTAGT	16318.9
NC_007793.1	61025	G	A	12068.7
NC_007793.1	150825	G	C	21829.8
NC_007793.1	164912	G	A	24032.8
NC_007793.1	240358	T	A	22131.2
NC_007793.1	270510	T	C	27653.4
NC_007793.1	292738	A	C	20607.6
NC_007793.1	376057	C	T	22861.6
NC_007793.1	376231	T	G	25360
NC_007793.1	425833	A	T	22861.4
NC_007793.1	460961	ATACAGC	AC	16624.5
NC_007793.1	514487	C	T	18551.9
NC_007793.1	514557	TTATGAAAATAAAGCAGT	TT	31929.1
NC_007793.1	517837	C	T	9107.48
NC_007793.1	517960	CTTTTTTTTATG	CTTTTTTTATG	16682.9
NC_007793.1	518090	T	C	36366.3
NC_007793.1	518102	G	A	37105.5
NC_007793.1	556082	T	C	39205
NC_007793.1	556291	T	C	21396.4
NC_007793.1	558118	T	C	13107.6
NC_007793.1	591624	G	C	23230.9
NC_007793.1	593858	ATA	AA	19935.4
NC_007793.1	656344	AATAG	TTTAT	15824.2
NC_007793.1	658463	AAT	TAA	15407.6
NC_007793.1	65866

In [50]:
%%bash
echo "=== FINAL FILTERED VARIANTS ==="
echo "Criteria: QUAL>1000, AF=1, DP>10"
echo ""

grep -v "^#" /content/variants/variants.vcf | \
  awk -F'\t' '$6 > 1000 && $8 ~ /AF=1;/' | \
  awk -F'\t' '{
    split($8, info, ";");
    for (i in info) {
      if (info[i] ~ /^DP=/) dp=info[i];
      if (info[i] ~ /^TYPE=/) type=info[i];
    }
    print $2"\t"$4"\t"$5"\t"$6"\t"dp"\t"type
  }' | \
  awk -F'\t' 'BEGIN{print "POS\tREF\tALT\tQUAL\tDP\tTYPE"} {print}'

echo ""
echo "Count:"
grep -v "^#" /content/variants/variants.vcf | \
  awk -F'\t' '$6 > 1000 && $8 ~ /AF=1;/' | wc -l

=== FINAL FILTERED VARIANTS ===
Criteria: QUAL>1000, AF=1, DP>10

POS	REF	ALT	QUAL	DP	TYPE
317	A	G	24626.5	DP=689	TYPE=snp
5010	C	G	29210.6	DP=822	TYPE=snp
34179	T	A	30645	DP=858	TYPE=snp
41290	ATTTTTTTAGT	ATTTTTTTTAGT	16318.9	DP=483	TYPE=ins
61025	G	A	12068.7	DP=342	TYPE=snp
150825	G	C	21829.8	DP=611	TYPE=snp
164912	G	A	24032.8	DP=680	TYPE=snp
240358	T	A	22131.2	DP=620	TYPE=snp
270510	T	C	27653.4	DP=773	TYPE=snp
292738	A	C	20607.6	DP=577	TYPE=snp
376057	C	T	22861.6	DP=641	TYPE=snp
376231	T	G	25360	DP=710	TYPE=snp
425833	A	T	22861.4	DP=642	TYPE=snp
460961	ATACAGC	AC	16624.5	DP=483	TYPE=del
514487	C	T	18551.9	DP=2649	TYPE=snp
514557	TTATGAAAATAAAGCAGT	TT	31929.1	DP=3575	TYPE=del
517837	C	T	9107.48	DP=432	TYPE=snp
517960	CTTTTTTTTATG	CTTTTTTTATG	16682.9	DP=739	TYPE=del
518090	T	C	36366.3	DP=1241	TYPE=snp
518102	G	A	37105.5	DP=1236	TYPE=snp
556082	T	C	39205	DP=1103	TYPE=snp
556291	T	C	21396.4	DP=753	TYPE=snp
558118	T	C	13107.6	DP=1811	TYPE=snp
591624	G	C	23230.9	DP=651	TYPE=snp
593858	ATA

In [51]:
%%bash
grep -v "^#" /content/variants/variants.vcf | \
  awk -F'\t' '$6 > 1000 && $8 ~ /AF=1;/' | \
  awk -F'\t' '{
    split($8, info, ";");
    for (i in info) {
      if (info[i] ~ /^DP=/) dp=info[i];
      if (info[i] ~ /^TYPE=/) type=info[i];
    }
    print $2"\t"$4"\t"$5"\t"$6"\t"dp"\t"type
  }' | \
  awk -F'\t' 'BEGIN{print "POS\tREF\tALT\tQUAL\tDP\tTYPE"} {print}' \
  > /content/variants/filtered_variants.tsv

echo "Saved. Preview:"
head -5 /content/variants/filtered_variants.tsv
wc -l /content/variants/filtered_variants.tsv

Saved. Preview:
POS	REF	ALT	QUAL	DP	TYPE
317	A	G	24626.5	DP=689	TYPE=snp
5010	C	G	29210.6	DP=822	TYPE=snp
34179	T	A	30645	DP=858	TYPE=snp
41290	ATTTTTTTAGT	ATTTTTTTTAGT	16318.9	DP=483	TYPE=ins
64 /content/variants/filtered_variants.tsv


## Final Results

- Reads mapped: **99.88%**
- Properly paired: **99.19%**
- Raw FreeBayes VCF records: **465**
- Filtered variant calls: **63**

**Note:** Duplicate marking was not performed in this workflow. The filtered calls are computational variant calls and were not experimentally validated.
